# Percobaan 7 - Reconstructed Features + Outcome Classifier

Notebook ini lanjut dari Percobaan 6 yang sudah naik ke sekitar 2.90.

Tambahan utama:
- `CatBoostClassifier` untuk win/draw/loss,
- classifier-aware score selection,
- submission baru tanpa menimpa hasil Percobaan 6.

In [1]:
import warnings
warnings.filterwarnings("ignore")

from collections import defaultdict, deque
from pathlib import Path
import time

import numpy as np
import pandas as pd
from catboost import CatBoostRegressor, CatBoostClassifier

pd.set_option("display.max_columns", 180)
pd.set_option("display.width", 220)

## 1. Path dan konfigurasi

Default `TASK_TYPE='CPU'` supaya stabil. Kalau CatBoost GPU di kernel `py_gpu_ready` aman, boleh ganti ke `GPU`.

In [2]:
BASE_PATH = Path.home() / "Downloads" / "Gammafest"
DATA_PATH = BASE_PATH / "dataset"
OUTPUT_DIR = BASE_PATH / "experiments" / "percobaan 7 - outcome classifier"

TRAIN_PATH = DATA_PATH / "train.csv"
TEST_PATH = DATA_PATH / "test.csv"
SAMPLE_PATH = DATA_PATH / "sample submission.csv"

SUBMISSION_PATH = OUTPUT_DIR / "submission_recon_outcome_classifier_jalur_a.csv"
SUBMISSION_ROUNDCLIP_PATH = OUTPUT_DIR / "submission_recon_outcome_classifier_roundclip.csv"
VALID_REPORT_PATH = OUTPUT_DIR / "validation_recon_outcome_regression_report.csv"
POSTPROCESS_REPORT_PATH = OUTPUT_DIR / "postprocess_recon_outcome_classifier_report.csv"

RANDOM_STATE = 42
VALID_FRAC = 0.20
TASK_TYPE = "GPU"
MAX_SCORE = 6
N_RANDOM_BLENDS = 2000

ELO_INIT = 1500.0
ELO_K_DEFAULT = 20
ELO_K_IMPORTANT = 40
IMPORTANT_TOURNAMENTS = {
    "FIFA World Cup", "AFC Asian Cup", "AFC Championship", "UEFA Euro",
    "Africa Cup of Nations", "African Cup of Nations", "Copa America", "Copa América",
    "Gold Cup", "CONCACAF Gold Cup"
}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Output:", OUTPUT_DIR)

Output: C:\Users\Gusti Jogish\Downloads\Gammafest\experiments\percobaan 7 - outcome classifier


## 2. Load data

In [3]:
train_raw = pd.read_csv(TRAIN_PATH)
test_raw = pd.read_csv(TEST_PATH)
sample = pd.read_csv(SAMPLE_PATH)

train_raw["date_dt"] = pd.to_datetime(train_raw["date"], errors="coerce")
test_raw["date_dt"] = pd.to_datetime(test_raw["date"], errors="coerce")

print("train:", train_raw.shape, train_raw["date_dt"].min(), "->", train_raw["date_dt"].max())
print("test :", test_raw.shape, test_raw["date_dt"].min(), "->", test_raw["date_dt"].max())
print("sample:", sample.shape)
display(train_raw.head(2))
display(test_raw.head(2))

train: (78772, 48) 1872-11-30 00:00:00 -> 2011-08-04 00:00:00
test : (42422, 21) 2011-08-06 00:00:00 -> 2026-03-31 00:00:00
sample: (42422, 3)


,Id,match_id,date,gender,team,opponent,is_home,neutral,tournament,venue_country,team_goals,opp_goals,team_points_last5,opp_points_last5,points_last5_diff,team_gd_last5,opp_gd_last5,gd_last5_diff,h2h_points_last5,h2h_gd_last5,days_since_last_match_team,days_since_last_match_opp,team_points_last10,opp_points_last10,team_avg_goals_last5,team_avg_conceded_last5,opp_avg_goals_last5,opp_avg_conceded_last5,team_win_rate_last10,opp_win_rate_last10,elo_team,elo_opponent,rank_team,rank_opponent,rank_diff,rank_missing_team,rank_missing_opp,confederation_team,confederation_opp,population_team,population_opp,gdp_per_capita_team,gdp_per_capita_opp,altitude_venue,distance_travel_team,distance_travel_opp,temperature_venue,date_dt
0,M000001_Scotland,M000001,1872-11-30,M,Scotland,England,1,0,Friendly,Scotland,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1500.0,1500.0,NaN,NaN,NaN,1,1,UEFA,UEFA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1872-11-30
1,M000001_England,M000001,1872-11-30,M,England,Scotland,0,0,Friendly,Scotland,0,0,NaN,1.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,0.0,NaN,1.0,NaN,NaN,0.0,0.0,NaN,0.0,1500.0,1500.0,NaN,NaN,NaN,1,1,UEFA,UEFA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1872-11-30


,Id,match_id,date,gender,team,opponent,is_home,neutral,tournament,venue_country,confederation_team,confederation_opp,population_team,population_opp,gdp_per_capita_team,gdp_per_capita_opp,altitude_venue,distance_travel_team,distance_travel_opp,temperature_venue,date_dt
0,M034984_Seychelles,M034984,2011-08-06,M,Seychelles,Mauritius,1,0,Indian Ocean Island Games,Seychelles,CAF,CAF,92409.0,1283330.0,12189.095160,9197.026972,-9999.0,0.000000,1751.895724,25.790169,2011-08-06
1,M034984_Mauritius,M034984,2011-08-06,M,Mauritius,Seychelles,0,0,Indian Ocean Island Games,Seychelles,CAF,CAF,1283330.0,92409.0,9197.026972,12189.095160,-9999.0,1751.895724,0.000000,25.790169,2011-08-06


## 3. Static features, temporal split, dan AW-MAE constants

In [4]:
CAT_COLS = [
    "gender", "team", "opponent", "tournament", "venue_country",
    "confederation_team", "confederation_opp",
]

STATIC_NUM_COLS = [
    "is_home", "neutral",
    "population_team", "population_opp",
    "gdp_per_capita_team", "gdp_per_capita_opp",
    "altitude_venue", "distance_travel_team", "distance_travel_opp", "temperature_venue",
]

DATE_FEATURES = ["year", "month", "dayofweek"]

TOURNAMENT_WEIGHTS = {
    "FIFA World Cup": 2.00,
    "AFC Championship": 1.80,
    "AFC Asian Cup": 1.80,
    "UEFA Euro": 1.80,
    "Copa America": 1.80,
    "Copa América": 1.80,
    "Africa Cup of Nations": 1.80,
    "African Cup of Nations": 1.80,
    "Gold Cup": 1.75,
    "CONCACAF Gold Cup": 1.75,
    "FIFA World Cup qualification": 1.50,
    "UEFA Euro qualification": 1.40,
    "AFC Asian Cup qualification": 1.40,
    "Friendly": 0.96,
}
DEFAULT_TOURNAMENT_WEIGHT = 1.20


def add_basic_features(df):
    df = df.copy()
    dt = pd.to_datetime(df["date"], errors="coerce")
    df["date_dt"] = dt
    df["year"] = dt.dt.year
    df["month"] = dt.dt.month
    df["dayofweek"] = dt.dt.dayofweek
    df["tournament_weight"] = df["tournament"].map(TOURNAMENT_WEIGHTS).fillna(DEFAULT_TOURNAMENT_WEIGHT)
    return df


match_dates = train_raw.groupby("match_id")["date_dt"].min().sort_values()
split_idx = int(len(match_dates) * (1 - VALID_FRAC))
train_match_ids = set(match_dates.index[:split_idx])
val_match_ids = set(match_dates.index[split_idx:])

tr_raw = train_raw[train_raw["match_id"].isin(train_match_ids)].copy()
val_raw = train_raw[train_raw["match_id"].isin(val_match_ids)].copy()

print("Train fold rows:", tr_raw.shape, tr_raw["date_dt"].min(), "->", tr_raw["date_dt"].max())
print("Valid fold rows:", val_raw.shape, val_raw["date_dt"].min(), "->", val_raw["date_dt"].max())
print("Train matches:", len(train_match_ids), "Valid matches:", len(val_match_ids))

Train fold rows: (63016, 48) 1872-11-30 00:00:00 -> 2005-01-30 00:00:00
Valid fold rows: (15756, 48) 2005-02-01 00:00:00 -> 2011-08-04 00:00:00
Train matches: 31508 Valid matches: 7878


## 4. Forward-only reconstruction

`can_update_outcome=True` hanya untuk history train yang boleh memperbarui state. Validation/test tidak memperbarui form, H2H, atau Elo dengan target mereka.
Schedule tetap memperbarui last-match date karena tanggal pertandingan memang diketahui.

In [5]:
def points_from_score(gf, ga):
    if gf > ga:
        return 3
    if gf == ga:
        return 1
    return 0


def elo_k(tournament):
    return ELO_K_IMPORTANT if tournament in IMPORTANT_TOURNAMENTS else ELO_K_DEFAULT


def summarize_form(history):
    if len(history) == 0:
        return {
            "points_last5": np.nan,
            "points_last10": np.nan,
            "gd_last5": np.nan,
            "avg_goals_last5": np.nan,
            "avg_conceded_last5": np.nan,
            "win_rate_last10": np.nan,
            "matches_known": 0,
        }
    last5 = list(history)[-5:]
    last10 = list(history)[-10:]
    return {
        "points_last5": float(sum(x["points"] for x in last5)),
        "points_last10": float(sum(x["points"] for x in last10)),
        "gd_last5": float(sum(x["gd"] for x in last5)),
        "avg_goals_last5": float(np.mean([x["gf"] for x in last5])),
        "avg_conceded_last5": float(np.mean([x["ga"] for x in last5])),
        "win_rate_last10": float(np.mean([x["points"] == 3 for x in last10])),
        "matches_known": int(len(history)),
    }


def build_reconstructed_features(input_df):
    df = add_basic_features(input_df).copy()
    df["_orig_order"] = np.arange(len(df))
    if "can_update_outcome" not in df.columns:
        df["can_update_outcome"] = df["team_goals"].notna() if "team_goals" in df.columns else False
    if "split_name" not in df.columns:
        df["split_name"] = "unknown"

    for col in ["team_goals", "opp_goals", "rank_team", "rank_opponent"]:
        if col not in df.columns:
            df[col] = np.nan

    df = df.sort_values(["date_dt", "match_id", "Id"]).reset_index(drop=True)

    form = defaultdict(lambda: deque(maxlen=50))
    h2h = defaultdict(lambda: deque(maxlen=20))
    elo = defaultdict(lambda: ELO_INIT)
    last_match_date = {}
    latest_rank = {}
    feature_rows = []

    for match_id, grp in df.groupby("match_id", sort=False):
        match_date = grp["date_dt"].iloc[0]

        for _, row in grp.iterrows():
            gender = row["gender"]
            team = row["team"]
            opp = row["opponent"]
            team_key = (gender, team)
            opp_key = (gender, opp)
            pair_key = (gender, team, opp)

            tf = summarize_form(form[team_key])
            of = summarize_form(form[opp_key])
            hhist = list(h2h[pair_key])[-5:]

            h2h_points = float(sum(x["points"] for x in hhist)) if hhist else np.nan
            h2h_gd = float(sum(x["gd"] for x in hhist)) if hhist else np.nan
            h2h_matches = int(len(hhist))

            team_last_date = last_match_date.get(team_key)
            opp_last_date = last_match_date.get(opp_key)
            days_team = (match_date - team_last_date).days if team_last_date is not None and pd.notna(match_date) else np.nan
            days_opp = (match_date - opp_last_date).days if opp_last_date is not None and pd.notna(match_date) else np.nan

            rank_team_latest = latest_rank.get(team_key, np.nan)
            rank_opp_latest = latest_rank.get(opp_key, np.nan)

            feature_rows.append({
                "_orig_order": row["_orig_order"],
                "team_points_last5_recon": tf["points_last5"],
                "opp_points_last5_recon": of["points_last5"],
                "points_last5_diff_recon": tf["points_last5"] - of["points_last5"] if pd.notna(tf["points_last5"]) and pd.notna(of["points_last5"]) else np.nan,
                "team_points_last10_recon": tf["points_last10"],
                "opp_points_last10_recon": of["points_last10"],
                "points_last10_diff_recon": tf["points_last10"] - of["points_last10"] if pd.notna(tf["points_last10"]) and pd.notna(of["points_last10"]) else np.nan,
                "team_gd_last5_recon": tf["gd_last5"],
                "opp_gd_last5_recon": of["gd_last5"],
                "gd_last5_diff_recon": tf["gd_last5"] - of["gd_last5"] if pd.notna(tf["gd_last5"]) and pd.notna(of["gd_last5"]) else np.nan,
                "team_avg_goals_last5_recon": tf["avg_goals_last5"],
                "team_avg_conceded_last5_recon": tf["avg_conceded_last5"],
                "opp_avg_goals_last5_recon": of["avg_goals_last5"],
                "opp_avg_conceded_last5_recon": of["avg_conceded_last5"],
                "avg_goals_diff_recon": tf["avg_goals_last5"] - of["avg_goals_last5"] if pd.notna(tf["avg_goals_last5"]) and pd.notna(of["avg_goals_last5"]) else np.nan,
                "avg_conceded_diff_recon": tf["avg_conceded_last5"] - of["avg_conceded_last5"] if pd.notna(tf["avg_conceded_last5"]) and pd.notna(of["avg_conceded_last5"]) else np.nan,
                "team_win_rate_last10_recon": tf["win_rate_last10"],
                "opp_win_rate_last10_recon": of["win_rate_last10"],
                "win_rate_diff_recon": tf["win_rate_last10"] - of["win_rate_last10"] if pd.notna(tf["win_rate_last10"]) and pd.notna(of["win_rate_last10"]) else np.nan,
                "h2h_points_last5_recon": h2h_points,
                "h2h_gd_last5_recon": h2h_gd,
                "h2h_matches_last5_recon": h2h_matches,
                "days_since_last_match_team_recon": days_team,
                "days_since_last_match_opp_recon": days_opp,
                "days_since_last_match_diff_recon": days_team - days_opp if pd.notna(days_team) and pd.notna(days_opp) else np.nan,
                "elo_team_recon": float(elo[team_key]),
                "elo_opponent_recon": float(elo[opp_key]),
                "elo_diff_recon": float(elo[team_key] - elo[opp_key]),
                "rank_team_latest_recon": rank_team_latest,
                "rank_opponent_latest_recon": rank_opp_latest,
                "rank_diff_latest_recon": rank_team_latest - rank_opp_latest if pd.notna(rank_team_latest) and pd.notna(rank_opp_latest) else np.nan,
                "rank_missing_team_recon": int(pd.isna(rank_team_latest)),
                "rank_missing_opp_recon": int(pd.isna(rank_opp_latest)),
                "team_matches_known_recon": tf["matches_known"],
                "opp_matches_known_recon": of["matches_known"],
                "matches_known_diff_recon": tf["matches_known"] - of["matches_known"],
            })

        for _, row in grp.iterrows():
            last_match_date[(row["gender"], row["team"])] = match_date

        can_update = bool(grp["can_update_outcome"].fillna(False).all())
        has_scores = grp["team_goals"].notna().all() and grp["opp_goals"].notna().all()
        if can_update and has_scores:
            for _, row in grp.iterrows():
                gender = row["gender"]
                team = row["team"]
                opp = row["opponent"]
                gf = float(row["team_goals"])
                ga = float(row["opp_goals"])
                pts = points_from_score(gf, ga)
                gd = gf - ga
                team_key = (gender, team)
                form[team_key].append({"points": pts, "gf": gf, "ga": ga, "gd": gd})
                h2h[(gender, team, opp)].append({"points": pts, "gd": gd})

                if pd.notna(row.get("rank_team", np.nan)):
                    latest_rank[team_key] = float(row["rank_team"])
                if pd.notna(row.get("rank_opponent", np.nan)):
                    latest_rank[(gender, opp)] = float(row["rank_opponent"])

            row = grp.iloc[0]
            gender = row["gender"]
            team_key = (gender, row["team"])
            opp_key = (gender, row["opponent"])
            e_team = elo[team_key]
            e_opp = elo[opp_key]
            expected_team = 1 / (1 + 10 ** ((e_opp - e_team) / 400))
            gf = float(row["team_goals"])
            ga = float(row["opp_goals"])
            actual_team = 1.0 if gf > ga else 0.5 if gf == ga else 0.0
            k = elo_k(row["tournament"])
            elo[team_key] = e_team + k * (actual_team - expected_team)
            elo[opp_key] = e_opp + k * ((1 - actual_team) - (1 - expected_team))

    feature_df = pd.DataFrame(feature_rows)
    return df.merge(feature_df, on="_orig_order", how="left").sort_values("_orig_order").reset_index(drop=True)

## 5. Build features untuk eval dan final test

In [6]:
tr_for_eval = tr_raw.copy()
tr_for_eval["split_name"] = "train_fold"
tr_for_eval["can_update_outcome"] = True

val_for_eval = val_raw.copy()
val_for_eval["split_name"] = "valid_fold"
val_for_eval["can_update_outcome"] = False

eval_input = pd.concat([tr_for_eval, val_for_eval], ignore_index=True, sort=False)
print("Building eval reconstructed features...")
eval_recon = build_reconstructed_features(eval_input)
tr_recon = eval_recon[eval_recon["split_name"] == "train_fold"].copy()
val_recon = eval_recon[eval_recon["split_name"] == "valid_fold"].copy()

full_train_for_final = train_raw.copy()
full_train_for_final["split_name"] = "full_train"
full_train_for_final["can_update_outcome"] = True

test_for_final = test_raw.copy()
test_for_final["split_name"] = "test"
test_for_final["can_update_outcome"] = False
test_for_final["team_goals"] = np.nan
test_for_final["opp_goals"] = np.nan

final_input = pd.concat([full_train_for_final, test_for_final], ignore_index=True, sort=False)
print("Building final reconstructed features...")
final_recon = build_reconstructed_features(final_input)
full_train_recon = final_recon[final_recon["split_name"] == "full_train"].copy()
test_recon = final_recon[final_recon["split_name"] == "test"].copy()

print("tr_recon:", tr_recon.shape)
print("val_recon:", val_recon.shape)
print("full_train_recon:", full_train_recon.shape)
print("test_recon:", test_recon.shape)

Building eval reconstructed features...
Building final reconstructed features...
tr_recon: (63016, 90)
val_recon: (15756, 90)
full_train_recon: (78772, 90)
test_recon: (42422, 90)


## 6. Sanity check dan preprocessing matrix

In [7]:
RECON_FEATURES = [
    "team_points_last5_recon", "opp_points_last5_recon", "points_last5_diff_recon",
    "team_points_last10_recon", "opp_points_last10_recon", "points_last10_diff_recon",
    "team_gd_last5_recon", "opp_gd_last5_recon", "gd_last5_diff_recon",
    "team_avg_goals_last5_recon", "team_avg_conceded_last5_recon",
    "opp_avg_goals_last5_recon", "opp_avg_conceded_last5_recon",
    "avg_goals_diff_recon", "avg_conceded_diff_recon",
    "team_win_rate_last10_recon", "opp_win_rate_last10_recon", "win_rate_diff_recon",
    "h2h_points_last5_recon", "h2h_gd_last5_recon", "h2h_matches_last5_recon",
    "days_since_last_match_team_recon", "days_since_last_match_opp_recon", "days_since_last_match_diff_recon",
    "elo_team_recon", "elo_opponent_recon", "elo_diff_recon",
    "rank_team_latest_recon", "rank_opponent_latest_recon", "rank_diff_latest_recon",
    "rank_missing_team_recon", "rank_missing_opp_recon",
    "team_matches_known_recon", "opp_matches_known_recon", "matches_known_diff_recon",
]

compare_pairs = [
    ("team_points_last5", "team_points_last5_recon"),
    ("opp_points_last5", "opp_points_last5_recon"),
    ("team_points_last10", "team_points_last10_recon"),
    ("opp_points_last10", "opp_points_last10_recon"),
    ("team_gd_last5", "team_gd_last5_recon"),
    ("opp_gd_last5", "opp_gd_last5_recon"),
    ("team_avg_goals_last5", "team_avg_goals_last5_recon"),
    ("team_avg_conceded_last5", "team_avg_conceded_last5_recon"),
    ("opp_avg_goals_last5", "opp_avg_goals_last5_recon"),
    ("opp_avg_conceded_last5", "opp_avg_conceded_last5_recon"),
    ("team_win_rate_last10", "team_win_rate_last10_recon"),
    ("opp_win_rate_last10", "opp_win_rate_last10_recon"),
    ("h2h_points_last5", "h2h_points_last5_recon"),
    ("h2h_gd_last5", "h2h_gd_last5_recon"),
    ("elo_team", "elo_team_recon"),
    ("elo_opponent", "elo_opponent_recon"),
]

rows = []
for orig, recon in compare_pairs:
    if orig in full_train_recon.columns and recon in full_train_recon.columns:
        tmp = full_train_recon[[orig, recon]].dropna()
        corr = tmp[orig].corr(tmp[recon]) if len(tmp) > 2 else np.nan
        rows.append({"original": orig, "reconstructed": recon, "non_null_pairs": len(tmp), "corr": corr})
compare_df = pd.DataFrame(rows).sort_values("corr")
display(compare_df)

BASE_NUM_COLS = STATIC_NUM_COLS + DATE_FEATURES + ["tournament_weight"] + RECON_FEATURES


def finalize_feature_frames(train_df, other_df, fit_name="train"):
    train = train_df.copy()
    other = other_df.copy()

    for df in [train, other]:
        for col in CAT_COLS:
            df[col] = df[col].fillna("Unknown").astype(str)
        for col in BASE_NUM_COLS:
            df[col] = pd.to_numeric(df[col], errors="coerce")
            df.loc[df[col] == -9999, col] = np.nan

    missing_flag_cols = []
    for col in BASE_NUM_COLS:
        flag_col = f"{col}_missing"
        train[flag_col] = train[col].isna().astype(int)
        other[flag_col] = other[col].isna().astype(int)
        missing_flag_cols.append(flag_col)

    medians = train[BASE_NUM_COLS].median(numeric_only=True)
    for col in BASE_NUM_COLS:
        fill_value = medians[col]
        if pd.isna(fill_value):
            fill_value = 0
        train[col] = train[col].fillna(fill_value)
        other[col] = other[col].fillna(fill_value)

    feature_cols = CAT_COLS + BASE_NUM_COLS + missing_flag_cols
    cat_feature_indices = [feature_cols.index(c) for c in CAT_COLS]
    print(f"{fit_name}: features={len(feature_cols)}, missing train={train[feature_cols].isna().sum().sum()}, missing other={other[feature_cols].isna().sum().sum()}")
    return train, other, feature_cols, cat_feature_indices


tr_model, val_model, FEATURE_COLS, CAT_FEATURE_INDICES = finalize_feature_frames(tr_recon, val_recon, fit_name="eval")
full_train_model, test_model, FINAL_FEATURE_COLS, FINAL_CAT_FEATURE_INDICES = finalize_feature_frames(full_train_recon, test_recon, fit_name="final")

assert FEATURE_COLS == FINAL_FEATURE_COLS
assert CAT_FEATURE_INDICES == FINAL_CAT_FEATURE_INDICES

X_tr = tr_model[FEATURE_COLS]
X_val = val_model[FEATURE_COLS]
y_tr_team = tr_model["team_goals"]
y_tr_opp = tr_model["opp_goals"]
y_val_team = val_model["team_goals"]
y_val_opp = val_model["opp_goals"]

X_full = full_train_model[FEATURE_COLS]
y_full_team = full_train_model["team_goals"]
y_full_opp = full_train_model["opp_goals"]
X_test = test_model[FEATURE_COLS]

print("X_tr:", X_tr.shape, "X_val:", X_val.shape, "X_test:", X_test.shape)

,original,reconstructed,non_null_pairs,corr
8,opp_avg_goals_last5,opp_avg_goals_last5_recon,78289,0.800315
5,opp_gd_last5,opp_gd_last5_recon,78289,0.811221
9,opp_avg_conceded_last5,opp_avg_conceded_last5_recon,78289,0.824315
4,team_gd_last5,team_gd_last5_recon,78289,0.839995
1,opp_points_last5,opp_points_last5_recon,78289,0.841340
6,team_avg_goals_last5,team_avg_goals_last5_recon,78289,0.842844
7,team_avg_conceded_last5,team_avg_conceded_last5_recon,78289,0.843728
11,opp_win_rate_last10,opp_win_rate_last10_recon,78289,0.868597
3,opp_points_last10,opp_points_last10_recon,78289,0.875067
0,team_points_last5,team_points_last5_recon,78289,0.890021


eval: features=105, missing train=0, missing other=0
final: features=105, missing train=0, missing other=0
X_tr: (63016, 105) X_val: (15756, 105) X_test: (42422, 105)


## 7. AW-MAE dan post-processing helpers

In [8]:
def outcome_label(team_goals, opp_goals):
    diff = np.asarray(team_goals) - np.asarray(opp_goals)
    return np.where(diff > 0, 1, np.where(diff < 0, -1, 0))


def postprocess_round_clip(team_pred, opp_pred, max_score=6):
    team = np.rint(team_pred).astype(int)
    opp = np.rint(opp_pred).astype(int)
    return np.clip(team, 0, max_score), np.clip(opp, 0, max_score)


def compute_awmae(df_true, team_pred, opp_pred):
    true_team = df_true["team_goals"].to_numpy()
    true_opp = df_true["opp_goals"].to_numpy()
    pred_team = np.asarray(team_pred)
    pred_opp = np.asarray(opp_pred)

    mae = (np.abs(true_team - pred_team) + np.abs(true_opp - pred_opp)) / 2
    exact = ((true_team == pred_team) & (true_opp == pred_opp)).astype(int)
    outcome = (outcome_label(true_team, true_opp) == outcome_label(pred_team, pred_opp)).astype(int)
    gd = ((true_team - true_opp) == (pred_team - pred_opp)).astype(int)
    penalty = 0.30 * (1 - exact) + 0.25 * (1 - outcome) + 0.15 * (1 - gd)
    multiplier = np.where(outcome == 1, 1.0, 1.5)
    loss = ((mae + penalty) * multiplier) ** 1.5
    weights = df_true["tournament"].map(TOURNAMENT_WEIGHTS).fillna(DEFAULT_TOURNAMENT_WEIGHT).to_numpy()
    score = np.sum(loss * weights) / np.sum(weights)
    diag = {
        "AW-MAE": score,
        "MAE_raw_component": mae.mean(),
        "exact_acc": exact.mean(),
        "outcome_acc": outcome.mean(),
        "goal_diff_acc": gd.mean(),
    }
    return score, diag


def evaluate_raw_predictions(name, df_true, team_raw, opp_raw, max_score=MAX_SCORE):
    team_int, opp_int = postprocess_round_clip(team_raw, opp_raw, max_score=max_score)
    score, diag = compute_awmae(df_true, team_int, opp_int)
    return {"model": name, **diag}

## 8. Train CatBoost ensemble

In [9]:
CATBOOST_CONFIGS = [
    {
        "name": "recon_cat_mae_d7_seed42",
        "params": dict(loss_function="MAE", eval_metric="MAE", iterations=1600, learning_rate=0.040, depth=7, l2_leaf_reg=7, random_strength=1.0, bagging_temperature=0.5, random_seed=42),
    },
    {
        "name": "recon_cat_mae_d6_seed7",
        "params": dict(loss_function="MAE", eval_metric="MAE", iterations=1800, learning_rate=0.035, depth=6, l2_leaf_reg=9, random_strength=1.6, bagging_temperature=0.8, random_seed=7),
    },
    {
        "name": "recon_cat_mae_d8_seed99",
        "params": dict(loss_function="MAE", eval_metric="MAE", iterations=1400, learning_rate=0.035, depth=8, l2_leaf_reg=10, random_strength=0.8, bagging_temperature=0.3, random_seed=99),
    },
    {
        "name": "recon_cat_rmse_d7_seed123",
        "params": dict(loss_function="RMSE", eval_metric="MAE", iterations=1500, learning_rate=0.040, depth=7, l2_leaf_reg=8, random_strength=1.2, bagging_temperature=0.6, random_seed=123),
    },
]

BASE_CAT_PARAMS = dict(task_type=TASK_TYPE, verbose=150, allow_writing_files=False)

val_pred_bank = {}
trained_val_models = {}
reports = []

start = time.time()
for cfg in CATBOOST_CONFIGS:
    name = cfg["name"]
    params = {**BASE_CAT_PARAMS, **cfg["params"]}
    print("\n" + "=" * 90)
    print("Training", name)

    mt = CatBoostRegressor(**params)
    mo = CatBoostRegressor(**params)
    mt.fit(X_tr, y_tr_team, cat_features=CAT_FEATURE_INDICES, eval_set=(X_val, y_val_team), use_best_model=True, early_stopping_rounds=160)
    mo.fit(X_tr, y_tr_opp, cat_features=CAT_FEATURE_INDICES, eval_set=(X_val, y_val_opp), use_best_model=True, early_stopping_rounds=160)

    pred_t = np.clip(mt.predict(X_val), 0, None)
    pred_o = np.clip(mo.predict(X_val), 0, None)
    val_pred_bank[name] = (pred_t, pred_o)
    trained_val_models[name] = (mt, mo)

    report = evaluate_raw_predictions(name, val_model, pred_t, pred_o, max_score=MAX_SCORE)
    report["best_iter_team"] = mt.best_iteration_
    report["best_iter_opp"] = mo.best_iteration_
    reports.append(report)
    print(report)

print("Training minutes:", (time.time() - start) / 60)
reports_df = pd.DataFrame(reports).sort_values("AW-MAE")
reports_df.to_csv(VALID_REPORT_PATH, index=False)
display(reports_df)
print("Saved:", VALID_REPORT_PATH)


Training recon_cat_mae_d7_seed42


Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 1.1747215	test: 1.1324730	best: 1.1324730 (0)	total: 73ms	remaining: 1m 56s
150:	learn: 1.0764289	test: 1.0519559	best: 1.0519559 (150)	total: 3.05s	remaining: 29.3s
300:	learn: 1.0436785	test: 1.0319599	best: 1.0319599 (300)	total: 5.92s	remaining: 25.5s
450:	learn: 1.0284858	test: 1.0255014	best: 1.0254977 (449)	total: 8.78s	remaining: 22.4s
600:	learn: 1.0189926	test: 1.0229576	best: 1.0229576 (600)	total: 11.6s	remaining: 19.3s
750:	learn: 1.0123772	test: 1.0220191	best: 1.0220041 (748)	total: 14.5s	remaining: 16.3s
900:	learn: 1.0065710	test: 1.0218576	best: 1.0218432 (899)	total: 17.2s	remaining: 13.4s
1050:	learn: 1.0011882	test: 1.0208597	best: 1.0208438 (1048)	total: 20.1s	remaining: 10.5s
1200:	learn: 0.9961421	test: 1.0203164	best: 1.0203051 (1192)	total: 22.9s	remaining: 7.59s
1350:	learn: 0.9915984	test: 1.0198381	best: 1.0198182 (1339)	total: 25.7s	remaining: 4.73s
1500:	learn: 0.9870842	test: 1.0194656	best: 1.0194272 (1489)	total: 28.5s	remaining: 1.88s
1599:	

Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 1.1747479	test: 1.1325134	best: 1.1325134 (0)	total: 17ms	remaining: 27.2s
150:	learn: 1.0770789	test: 1.0521239	best: 1.0521239 (150)	total: 2.66s	remaining: 25.5s
300:	learn: 1.0444436	test: 1.0329329	best: 1.0329329 (300)	total: 5.29s	remaining: 22.8s
450:	learn: 1.0287080	test: 1.0255314	best: 1.0255314 (450)	total: 7.95s	remaining: 20.3s
600:	learn: 1.0197235	test: 1.0229583	best: 1.0229583 (600)	total: 10.7s	remaining: 17.7s
750:	learn: 1.0130423	test: 1.0217732	best: 1.0217620 (749)	total: 13.3s	remaining: 15.1s
900:	learn: 1.0067126	test: 1.0212055	best: 1.0211995 (899)	total: 16.1s	remaining: 12.5s
1050:	learn: 1.0015278	test: 1.0207471	best: 1.0207471 (1050)	total: 18.9s	remaining: 9.89s
1200:	learn: 0.9967525	test: 1.0202360	best: 1.0202284 (1199)	total: 21.6s	remaining: 7.19s
1350:	learn: 0.9921411	test: 1.0199393	best: 1.0199382 (1349)	total: 24.4s	remaining: 4.49s
1500:	learn: 0.9877070	test: 1.0197819	best: 1.0197481 (1493)	total: 27.1s	remaining: 1.79s
1599:	l

...

0:	learn: 1.2750193	test: 1.2666925	best: 1.2666925 (0)	total: 22ms	remaining: 33s
150:	learn: 1.0449129	test: 1.0741839	best: 1.0740737 (148)	total: 3.37s	remaining: 30.1s
300:	learn: 1.0261660	test: 1.0734326	best: 1.0730031 (254)	total: 6.59s	remaining: 26.3s
450:	learn: 1.0123414	test: 1.0719646	best: 1.0716323 (423)	total: 9.67s	remaining: 22.5s
600:	learn: 1.0008875	test: 1.0696244	best: 1.0696244 (600)	total: 12.8s	remaining: 19.1s
750:	learn: 0.9909679	test: 1.0691413	best: 1.0689067 (703)	total: 16s	remaining: 16s
bestTest = 1.068906662
bestIteration = 703
Shrink model to first 704 iterations.


Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 1.2745150	test: 1.2660930	best: 1.2660930 (0)	total: 20.4ms	remaining: 30.6s
150:	learn: 1.0449265	test: 1.0739299	best: 1.0735371 (111)	total: 3.06s	remaining: 27.3s
300:	learn: 1.0258933	test: 1.0733244	best: 1.0729356 (294)	total: 6.13s	remaining: 24.4s
450:	learn: 1.0123460	test: 1.0715696	best: 1.0714818 (448)	total: 9.25s	remaining: 21.5s
600:	learn: 0.9999715	test: 1.0709472	best: 1.0704420 (569)	total: 12.4s	remaining: 18.5s
bestTest = 1.070442039
bestIteration = 569
Shrink model to first 570 iterations.
{'model': 'recon_cat_rmse_d7_seed123', 'AW-MAE': np.float64(3.09165636500501), 'MAE_raw_component': np.float64(1.0384615384615385), 'exact_acc': np.float64(0.0965981213505966), 'outcome_acc': np.float64(0.5685453160700685), 'goal_diff_acc': np.float64(0.23032495557248034), 'best_iter_team': 703, 'best_iter_opp': 569}
Training minutes: 3.513915276527405


,model,AW-MAE,MAE_raw_component,exact_acc,outcome_acc,goal_diff_acc,best_iter_team,best_iter_opp
3,recon_cat_rmse_d7_seed123,3.091656,1.038462,0.096598,0.568545,0.230325,703,569
0,recon_cat_mae_d7_seed42,3.110370,0.996954,0.110942,0.512249,0.234641,1585,1599
1,recon_cat_mae_d6_seed7,3.131402,1.001206,0.109799,0.510663,0.233752,1758,1796
2,recon_cat_mae_d8_seed99,3.137529,1.001396,0.109546,0.504379,0.234768,895,971


Saved: C:\Users\Gusti Jogish\Downloads\Gammafest\experiments\percobaan 7 - outcome classifier\validation_recon_outcome_regression_report.csv


## 9. Blend search

In [10]:
model_names = list(val_pred_bank.keys())
team_matrix = np.vstack([val_pred_bank[name][0] for name in model_names])
opp_matrix = np.vstack([val_pred_bank[name][1] for name in model_names])


def score_blend(weights, max_score=MAX_SCORE):
    weights = np.asarray(weights, dtype=float)
    weights = weights / weights.sum()
    pred_t = np.average(team_matrix, axis=0, weights=weights)
    pred_o = np.average(opp_matrix, axis=0, weights=weights)
    pred_t_int, pred_o_int = postprocess_round_clip(pred_t, pred_o, max_score=max_score)
    score, diag = compute_awmae(val_model, pred_t_int, pred_o_int)
    return score, diag


blend_rows = []
best = {"score": np.inf, "weights": None, "name": None, "diag": None}

for i, name in enumerate(model_names):
    w = np.zeros(len(model_names)); w[i] = 1
    score, diag = score_blend(w)
    blend_rows.append({"blend": f"single_{name}", "AW-MAE": score, **{f"w_{n}": w[j] for j, n in enumerate(model_names)}})
    if score < best["score"]:
        best = {"score": score, "weights": w.copy(), "name": f"single_{name}", "diag": diag}

w = np.ones(len(model_names)) / len(model_names)
score, diag = score_blend(w)
blend_rows.append({"blend": "equal_average", "AW-MAE": score, **{f"w_{n}": w[j] for j, n in enumerate(model_names)}})
if score < best["score"]:
    best = {"score": score, "weights": w.copy(), "name": "equal_average", "diag": diag}

single_scores = []
for i, name in enumerate(model_names):
    w_single = np.zeros(len(model_names)); w_single[i] = 1
    s, _ = score_blend(w_single)
    single_scores.append(s)
w = 1 / np.maximum(np.array(single_scores), 1e-9)
w = w / w.sum()
score, diag = score_blend(w)
blend_rows.append({"blend": "inverse_awmae", "AW-MAE": score, **{f"w_{n}": w[j] for j, n in enumerate(model_names)}})
if score < best["score"]:
    best = {"score": score, "weights": w.copy(), "name": "inverse_awmae", "diag": diag}

rng = np.random.default_rng(RANDOM_STATE)
alpha_options = [0.25, 0.5, 1.0, 2.0]
for k in range(N_RANDOM_BLENDS):
    alpha = alpha_options[k % len(alpha_options)]
    w = rng.dirichlet(np.ones(len(model_names)) * alpha)
    score, diag = score_blend(w)
    if k < 25 or score < best["score"]:
        blend_rows.append({"blend": f"random_{k}_a{alpha}", "AW-MAE": score, **{f"w_{n}": w[j] for j, n in enumerate(model_names)}})
    if score < best["score"]:
        best = {"score": score, "weights": w.copy(), "name": f"random_{k}_a{alpha}", "diag": diag}

blend_df = pd.DataFrame(blend_rows).sort_values("AW-MAE").reset_index(drop=True)
display(blend_df.head(20))

print("Best blend:", best["name"])
print("Best AW-MAE:", best["score"])
print("Diagnostics:", best["diag"])
print("Weights:")
for name, weight in sorted(zip(model_names, best["weights"]), key=lambda x: -x[1]):
    print(f"  {name:<32} {weight:.5f}")

,blend,AW-MAE,w_recon_cat_mae_d7_seed42,w_recon_cat_mae_d6_seed7,w_recon_cat_mae_d8_seed99,w_recon_cat_rmse_d7_seed123
0,random_1124_a0.25,3.066247,0.089220,0.059679,0.164406,0.686694
1,random_216_a0.25,3.067868,0.383465,0.001930,0.001622,0.612983
2,random_118_a1.0,3.068787,0.317943,0.002083,0.113085,0.566889
3,random_48_a0.25,3.070073,0.001429,0.178225,0.123840,0.696505
4,random_26_a1.0,3.073277,0.129002,0.025946,0.061713,0.783338
5,random_9_a0.5,3.073394,0.197650,0.000592,0.028711,0.773046
6,random_5_a0.5,3.074283,0.026982,0.100146,0.299397,0.573474
7,single_recon_cat_rmse_d7_seed123,3.091656,0.000000,0.000000,0.000000,1.000000
8,random_1_a0.5,3.098078,0.017968,0.150514,0.490905,0.340613
9,random_22_a1.0,3.098103,0.398764,0.116891,0.179967,0.304377


Best blend: random_1124_a0.25
Best AW-MAE: 3.066246913122315
Diagnostics: {'AW-MAE': np.float64(3.066246913122315), 'MAE_raw_component': np.float64(1.016025641025641), 'exact_acc': np.float64(0.10148514851485149), 'outcome_acc': np.float64(0.5489337395277989), 'goal_diff_acc': np.float64(0.23343488194973344)}
Weights:
  recon_cat_rmse_d7_seed123        0.68669
  recon_cat_mae_d8_seed99          0.16441
  recon_cat_mae_d7_seed42          0.08922
  recon_cat_mae_d6_seed7           0.05968


## 10. Outcome classifier win/draw/loss

Regressor menebak jumlah gol, tapi AW-MAE sangat menghukum outcome yang salah.

Cell ini menambahkan beberapa `CatBoostClassifier` untuk memprediksi outcome dari perspektif row:
- `0`: team kalah,
- `1`: seri,
- `2`: team menang.

Probabilitas classifier akan dipakai di post-processing skor integer.

In [11]:
def make_outcome_target(df):
    diff = df["team_goals"].to_numpy() - df["opp_goals"].to_numpy()
    return np.where(diff > 0, 2, np.where(diff < 0, 0, 1)).astype(int)


def aligned_outcome_proba(model, X):
    raw = model.predict_proba(X)
    out = np.zeros((len(X), 3), dtype=float)
    for j, cls in enumerate(model.classes_):
        out[:, int(cls)] = raw[:, j]
    row_sum = out.sum(axis=1, keepdims=True)
    row_sum[row_sum == 0] = 1.0
    return out / row_sum


y_tr_outcome = make_outcome_target(tr_model)
y_val_outcome = make_outcome_target(val_model)
y_full_outcome = make_outcome_target(full_train_model)

CLASSIFIER_CONFIGS = [
    {
        "name": "outcome_cls_d6_seed42",
        "params": dict(loss_function="MultiClass", eval_metric="MultiClass", iterations=1200, learning_rate=0.045, depth=6, l2_leaf_reg=6, random_strength=1.0, random_seed=42),
    },
    {
        "name": "outcome_cls_d7_seed7",
        "params": dict(loss_function="MultiClass", eval_metric="MultiClass", iterations=1000, learning_rate=0.040, depth=7, l2_leaf_reg=8, random_strength=1.4, random_seed=7),
    },
    {
        "name": "outcome_cls_d5_seed99",
        "params": dict(loss_function="MultiClass", eval_metric="MultiClass", iterations=1400, learning_rate=0.035, depth=5, l2_leaf_reg=5, random_strength=1.8, random_seed=99),
    },
]

BASE_CLS_PARAMS = dict(task_type=TASK_TYPE, verbose=150, allow_writing_files=False)

outcome_val_proba_bank = {}
trained_val_classifiers = {}
cls_reports = []

start = time.time()
for cfg in CLASSIFIER_CONFIGS:
    name = cfg["name"]
    params = {**BASE_CLS_PARAMS, **cfg["params"]}
    print("\n" + "=" * 90)
    print("Training", name)

    clf = CatBoostClassifier(**params)
    clf.fit(
        X_tr,
        y_tr_outcome,
        cat_features=CAT_FEATURE_INDICES,
        eval_set=(X_val, y_val_outcome),
        use_best_model=True,
        early_stopping_rounds=140,
    )

    proba = aligned_outcome_proba(clf, X_val)
    pred = proba.argmax(axis=1)
    acc = (pred == y_val_outcome).mean()
    nll = -np.log(np.clip(proba[np.arange(len(y_val_outcome)), y_val_outcome], 1e-12, 1)).mean()

    outcome_val_proba_bank[name] = proba
    trained_val_classifiers[name] = clf
    cls_reports.append({"classifier": name, "outcome_acc": acc, "nll": nll, "best_iter": clf.best_iteration_})
    print(cls_reports[-1])

cls_report_df = pd.DataFrame(cls_reports).sort_values(["nll", "outcome_acc"], ascending=[True, False])
display(cls_report_df)
print("Classifier training minutes:", (time.time() - start) / 60)

outcome_model_names = list(outcome_val_proba_bank.keys())
outcome_val_proba = np.mean([outcome_val_proba_bank[name] for name in outcome_model_names], axis=0)
outcome_val_pred = outcome_val_proba.argmax(axis=1)
print("Avg classifier outcome_acc:", (outcome_val_pred == y_val_outcome).mean())
print("Avg classifier nll:", -np.log(np.clip(outcome_val_proba[np.arange(len(y_val_outcome)), y_val_outcome], 1e-12, 1)).mean())


Training outcome_cls_d6_seed42
0:	learn: 1.0830115	test: 1.0820564	best: 1.0820564 (0)	total: 79.5ms	remaining: 1m 35s
150:	learn: 0.8786137	test: 0.8827267	best: 0.8827267 (150)	total: 1.22s	remaining: 8.48s
300:	learn: 0.8626314	test: 0.8789111	best: 0.8789111 (300)	total: 2.31s	remaining: 6.89s
450:	learn: 0.8523468	test: 0.8785469	best: 0.8784322 (399)	total: 3.35s	remaining: 5.56s
bestTest = 0.8784321623
bestIteration = 399
Shrink model to first 400 iterations.
{'classifier': 'outcome_cls_d6_seed42', 'outcome_acc': np.float64(0.5990733688753491), 'nll': np.float64(0.8784319889106345), 'best_iter': 399}

Training outcome_cls_d7_seed7
0:	learn: 1.0846232	test: 1.0838908	best: 1.0838908 (0)	total: 9.58ms	remaining: 9.57s
150:	learn: 0.8749490	test: 0.8810725	best: 0.8810725 (150)	total: 1.53s	remaining: 8.62s
300:	learn: 0.8579522	test: 0.8776855	best: 0.8776855 (300)	total: 3.24s	remaining: 7.52s
450:	learn: 0.8453869	test: 0.8768792	best: 0.8767651 (438)	total: 4.65s	remaining: 5.

,classifier,outcome_acc,nll,best_iter
1,outcome_cls_d7_seed7,0.599073,0.876765,438
2,outcome_cls_d5_seed99,0.598439,0.878171,586
0,outcome_cls_d6_seed42,0.599073,0.878432,399


Classifier training minutes: 0.27269800504048664
Avg classifier outcome_acc: 0.5991368367605991
Avg classifier nll: 0.8770972453683207


## 11. Classifier-aware score post-processing

Kita tetap memakai raw prediction dari regression ensemble, tapi kandidat skor integer sekarang diberi penalti kalau outcome-nya tidak didukung classifier.

Biaya kandidat skor:

```text
distance_to_regression
+ classifier_weight * -log(P(outcome kandidat))
+ gd_weight * distance_goal_difference
+ prior_weight * -log(P(skor historis))
```

In [12]:
def build_score_pair_prior(df, max_score=6, smoothing=1.0):
    counts = np.full((max_score + 1, max_score + 1), smoothing, dtype=float)
    team = np.clip(df["team_goals"].round().astype(int).to_numpy(), 0, max_score)
    opp = np.clip(df["opp_goals"].round().astype(int).to_numpy(), 0, max_score)
    for tg, og in zip(team, opp):
        counts[tg, og] += 1.0
    return counts / counts.sum()


def classifier_aware_postprocess(team_raw, opp_raw, outcome_proba, params, prior_matrix=None):
    max_score = int(params.get("max_score", 6))
    classifier_weight = float(params.get("classifier_weight", 0.20))
    gd_weight = float(params.get("gd_weight", 0.05))
    prior_weight = float(params.get("prior_weight", 0.02))

    team_raw = np.asarray(team_raw, dtype=float)
    opp_raw = np.asarray(opp_raw, dtype=float)
    outcome_proba = np.asarray(outcome_proba, dtype=float)

    candidates = np.array([(tg, og) for tg in range(max_score + 1) for og in range(max_score + 1)], dtype=int)
    cand_team = candidates[:, 0]
    cand_opp = candidates[:, 1]
    cand_diff = cand_team - cand_opp
    # Class order: 0 lose, 1 draw, 2 win.
    cand_outcome_class = np.where(cand_diff > 0, 2, np.where(cand_diff < 0, 0, 1))

    base_cost = (np.abs(team_raw[:, None] - cand_team[None, :]) + np.abs(opp_raw[:, None] - cand_opp[None, :])) / 2
    gd_cost = np.abs((team_raw - opp_raw)[:, None] - cand_diff[None, :])
    outcome_cost = -np.log(np.clip(outcome_proba[:, cand_outcome_class], 1e-12, 1.0))

    total_cost = base_cost + classifier_weight * outcome_cost + gd_weight * gd_cost

    if prior_matrix is not None and prior_weight > 0:
        prior = np.clip(prior_matrix[cand_team, cand_opp], 1e-12, None)
        total_cost = total_cost + prior_weight * (-np.log(prior))[None, :]

    best_idx = np.argmin(total_cost, axis=1)
    return cand_team[best_idx].astype(int), cand_opp[best_idx].astype(int)


blend_weights = best["weights"] / best["weights"].sum()
val_team_raw = np.average(team_matrix, axis=0, weights=blend_weights)
val_opp_raw = np.average(opp_matrix, axis=0, weights=blend_weights)

clip_rows = []
for max_score in range(4, 9):
    pred_t, pred_o = postprocess_round_clip(val_team_raw, val_opp_raw, max_score=max_score)
    score, diag = compute_awmae(val_model, pred_t, pred_o)
    clip_rows.append({"max_score": max_score, **diag})
clip_df = pd.DataFrame(clip_rows).sort_values("AW-MAE")
display(clip_df)

base_max_score = int(clip_df.iloc[0]["max_score"])
base_pred_t, base_pred_o = postprocess_round_clip(val_team_raw, val_opp_raw, max_score=base_max_score)
base_score, base_diag = compute_awmae(val_model, base_pred_t, base_pred_o)
print("Round+clip best:", base_score, "max_score:", base_max_score)

max_score_candidates = sorted(set([base_max_score, max(4, base_max_score - 1), min(8, base_max_score + 1)]))
classifier_weights = [0.00, 0.05, 0.10, 0.20, 0.35, 0.50, 0.75]
gd_weights = [0.00, 0.03, 0.06, 0.10]
prior_weights = [0.00, 0.01, 0.03, 0.06]

pp_rows = []
best_pp = {
    "score": base_score,
    "params": {"mode": "round_clip", "max_score": base_max_score, "classifier_weight": 0.0, "gd_weight": 0.0, "prior_weight": 0.0},
    "diag": base_diag,
}

start = time.time()
for max_score in max_score_candidates:
    prior_matrix = build_score_pair_prior(tr_model, max_score=max_score, smoothing=1.0)
    for classifier_weight in classifier_weights:
        for gd_weight in gd_weights:
            for prior_weight in prior_weights:
                params = {
                    "mode": "classifier_aware",
                    "max_score": max_score,
                    "classifier_weight": classifier_weight,
                    "gd_weight": gd_weight,
                    "prior_weight": prior_weight,
                }
                pred_t, pred_o = classifier_aware_postprocess(val_team_raw, val_opp_raw, outcome_val_proba, params, prior_matrix=prior_matrix)
                score, diag = compute_awmae(val_model, pred_t, pred_o)
                pp_rows.append({**params, **diag})
                if score < best_pp["score"]:
                    best_pp = {"score": score, "params": params.copy(), "diag": diag.copy()}

pp_df = pd.DataFrame(pp_rows).sort_values("AW-MAE").reset_index(drop=True)
pp_df.to_csv(POSTPROCESS_REPORT_PATH, index=False)
BEST_PP_PARAMS = best_pp["params"]
BEST_PP_SCORE = best_pp["score"]
BEST_MAX_SCORE = int(BEST_PP_PARAMS["max_score"])

print("Postprocess search minutes:", (time.time() - start) / 60)
print("Best classifier-aware score:", BEST_PP_SCORE)
print("Best classifier-aware params:", BEST_PP_PARAMS)
print("Best diagnostics:", best_pp["diag"])
display(pp_df.head(20))

,max_score,AW-MAE,MAE_raw_component,exact_acc,outcome_acc,goal_diff_acc
0,4,3.054944,1.013265,0.103008,0.548934,0.235402
1,5,3.061995,1.014947,0.100787,0.548934,0.232673
2,6,3.066247,1.016026,0.101485,0.548934,0.233435
3,7,3.071649,1.018025,0.100343,0.548934,0.232483
4,8,3.074947,1.019167,0.100406,0.548934,0.232546


Round+clip best: 3.054943869878199 max_score: 4
Postprocess search minutes: 0.07682281732559204
Best classifier-aware score: 2.9825615375092593
Best classifier-aware params: {'mode': 'classifier_aware', 'max_score': 4, 'classifier_weight': 0.75, 'gd_weight': 0.1, 'prior_weight': 0.06}
Best diagnostics: {'AW-MAE': np.float64(2.9825615375092593), 'MAE_raw_component': np.float64(1.0226580350342727), 'exact_acc': np.float64(0.09958111195734957), 'outcome_acc': np.float64(0.5919649657273419), 'goal_diff_acc': np.float64(0.2303884234577304)}


,mode,max_score,classifier_weight,gd_weight,prior_weight,AW-MAE,MAE_raw_component,exact_acc,outcome_acc,goal_diff_acc
0,classifier_aware,4,0.75,0.10,0.06,2.982562,1.022658,0.099581,0.591965,0.230388
1,classifier_aware,4,0.50,0.00,0.01,2.982735,1.022880,0.100089,0.590061,0.233118
2,classifier_aware,4,0.50,0.00,0.00,2.983358,1.023547,0.099708,0.590378,0.233054
3,classifier_aware,4,0.50,0.00,0.06,2.983372,1.021198,0.101041,0.587903,0.233625
4,classifier_aware,4,0.50,0.03,0.06,2.983723,1.020881,0.101295,0.587459,0.233816
5,classifier_aware,4,0.50,0.00,0.03,2.984826,1.022912,0.099581,0.588982,0.232292
6,classifier_aware,4,0.50,0.10,0.06,2.986057,1.019358,0.100660,0.585872,0.232166
7,classifier_aware,4,0.75,0.06,0.06,2.986881,1.024213,0.099772,0.591775,0.230515
8,classifier_aware,4,0.50,0.06,0.06,2.987336,1.020564,0.101295,0.586888,0.233308
9,classifier_aware,4,0.75,0.10,0.03,2.987392,1.024150,0.099200,0.591711,0.229183


## 12. Train final regressors/classifiers dan generate submission

In [13]:
final_pred_bank = {}
final_models = {}

for cfg in CATBOOST_CONFIGS:
    name = cfg["name"]
    val_t, val_o = trained_val_models[name]
    best_iter = int(max(getattr(val_t, "best_iteration_", 800), getattr(val_o, "best_iteration_", 800)) + 140)
    params = {**BASE_CAT_PARAMS, **cfg["params"]}
    params["iterations"] = max(best_iter, 350)
    params.pop("eval_metric", None)

    print("\n" + "=" * 90)
    print("Final regression training", name, "iterations:", params["iterations"])
    mt = CatBoostRegressor(**params)
    mo = CatBoostRegressor(**params)
    mt.fit(X_full, y_full_team, cat_features=CAT_FEATURE_INDICES, verbose=150)
    mo.fit(X_full, y_full_opp, cat_features=CAT_FEATURE_INDICES, verbose=150)

    final_pred_bank[name] = (np.clip(mt.predict(X_test), 0, None), np.clip(mo.predict(X_test), 0, None))
    final_models[name] = (mt, mo)

final_outcome_probas = []
final_classifiers = {}

for cfg in CLASSIFIER_CONFIGS:
    name = cfg["name"]
    val_clf = trained_val_classifiers[name]
    best_iter = int(getattr(val_clf, "best_iteration_", 700) + 120)
    params = {**BASE_CLS_PARAMS, **cfg["params"]}
    params["iterations"] = max(best_iter, 250)
    params.pop("eval_metric", None)

    print("\n" + "=" * 90)
    print("Final classifier training", name, "iterations:", params["iterations"])
    clf = CatBoostClassifier(**params)
    clf.fit(X_full, y_full_outcome, cat_features=CAT_FEATURE_INDICES, verbose=150)
    proba = aligned_outcome_proba(clf, X_test)
    final_outcome_probas.append(proba)
    final_classifiers[name] = clf

test_outcome_proba = np.mean(final_outcome_probas, axis=0)

test_team_matrix = np.vstack([final_pred_bank[name][0] for name in model_names])
test_opp_matrix = np.vstack([final_pred_bank[name][1] for name in model_names])
weights = best["weights"] / best["weights"].sum()
test_team_raw = np.average(test_team_matrix, axis=0, weights=weights)
test_opp_raw = np.average(test_opp_matrix, axis=0, weights=weights)

round_team, round_opp = postprocess_round_clip(test_team_raw, test_opp_raw, max_score=BEST_MAX_SCORE)
submission_round = sample[["Id"]].copy()
submission_round["team_goals"] = round_team
submission_round["opp_goals"] = round_opp
submission_round.to_csv(SUBMISSION_ROUNDCLIP_PATH, index=False)

if BEST_PP_PARAMS.get("mode") == "classifier_aware":
    prior_matrix = build_score_pair_prior(full_train_model, max_score=int(BEST_PP_PARAMS["max_score"]), smoothing=1.0)
    final_team, final_opp = classifier_aware_postprocess(test_team_raw, test_opp_raw, test_outcome_proba, BEST_PP_PARAMS, prior_matrix=prior_matrix)
    active_mode = "classifier_aware"
else:
    final_team, final_opp = round_team, round_opp
    active_mode = "round_clip"

submission = sample[["Id"]].copy()
submission["team_goals"] = final_team
submission["opp_goals"] = final_opp
assert submission.shape == sample.shape
assert submission["Id"].equals(sample["Id"])
submission.to_csv(SUBMISSION_PATH, index=False)

changed_rows = ((submission["team_goals"] != submission_round["team_goals"]) | (submission["opp_goals"] != submission_round["opp_goals"])).sum()

print("Saved submission:", SUBMISSION_PATH)
print("Saved round+clip backup:", SUBMISSION_ROUNDCLIP_PATH)
print("Active mode:", active_mode)
print("Validation best blend AW-MAE:", best["score"])
print("Validation best classifier-aware AW-MAE:", BEST_PP_SCORE)
print("Best params:", BEST_PP_PARAMS)
print("Changed rows vs round+clip:", changed_rows)
print("Shape:", submission.shape)

display(submission.head())
display(submission[["team_goals", "opp_goals"]].describe())

print("\ndistribusi skor")
print("\nteam_goals")
print(submission["team_goals"].value_counts())
print("\nopp_goals")
print(submission["opp_goals"].value_counts())

print("\ndistribusi pasangan skor")
display(submission[["team_goals", "opp_goals"]].value_counts().head(30).to_frame("count"))


Final regression training recon_cat_mae_d7_seed42 iterations: 1739


Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 1.1661994	total: 24.5ms	remaining: 42.6s
150:	learn: 1.0657903	total: 2.9s	remaining: 30.5s
300:	learn: 1.0346090	total: 5.84s	remaining: 27.9s
450:	learn: 1.0206317	total: 8.67s	remaining: 24.8s
600:	learn: 1.0121013	total: 11.6s	remaining: 22s
750:	learn: 1.0061712	total: 14.5s	remaining: 19.1s
900:	learn: 1.0005558	total: 17.3s	remaining: 16.1s
1050:	learn: 0.9953752	total: 20.2s	remaining: 13.3s
1200:	learn: 0.9909512	total: 23.2s	remaining: 10.4s
1350:	learn: 0.9870380	total: 26.3s	remaining: 7.54s
1500:	learn: 0.9830700	total: 29.1s	remaining: 4.62s
1650:	learn: 0.9792761	total: 31.9s	remaining: 1.7s
1738:	learn: 0.9773054	total: 33.6s	remaining: 0us


Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 1.1662432	total: 17.8ms	remaining: 30.9s
150:	learn: 1.0658667	total: 2.7s	remaining: 28.4s
300:	learn: 1.0347086	total: 5.56s	remaining: 26.6s
450:	learn: 1.0210968	total: 8.34s	remaining: 23.8s
600:	learn: 1.0122466	total: 11.1s	remaining: 21s
750:	learn: 1.0064113	total: 13.8s	remaining: 18.2s
900:	learn: 1.0003985	total: 16.6s	remaining: 15.4s
1050:	learn: 0.9952989	total: 19.4s	remaining: 12.7s
1200:	learn: 0.9908904	total: 22.1s	remaining: 9.92s
1350:	learn: 0.9872928	total: 24.9s	remaining: 7.16s
1500:	learn: 0.9834156	total: 27.7s	remaining: 4.4s
1650:	learn: 0.9798721	total: 30.6s	remaining: 1.63s
1738:	learn: 0.9778794	total: 32.3s	remaining: 0us

Final regression training recon_cat_mae_d6_seed7 iterations: 1936


...

0:	learn: 1.1661217	total: 20.9ms	remaining: 23.2s
150:	learn: 1.0700301	total: 3.27s	remaining: 20.8s
300:	learn: 1.0353580	total: 6.54s	remaining: 17.6s
450:	learn: 1.0196145	total: 9.88s	remaining: 14.5s
600:	learn: 1.0101053	total: 13.1s	remaining: 11.1s
750:	learn: 1.0032033	total: 16.4s	remaining: 7.84s
900:	learn: 0.9977478	total: 19.6s	remaining: 4.58s
1050:	learn: 0.9919985	total: 23s	remaining: 1.31s
1110:	learn: 0.9897929	total: 24.4s	remaining: 0us

Final regression training recon_cat_rmse_d7_seed123 iterations: 843
0:	learn: 1.7756212	total: 17.1ms	remaining: 14.4s
150:	learn: 1.4389151	total: 2.83s	remaining: 13s
300:	learn: 1.4024274	total: 6.29s	remaining: 11.3s
450:	learn: 1.3775230	total: 10.6s	remaining: 9.18s
600:	learn: 1.3575928	total: 14.7s	remaining: 5.91s
750:	learn: 1.3416695	total: 19.1s	remaining: 2.34s
842:	learn: 1.3322598	total: 21.8s	remaining: 0us
0:	learn: 1.7754276	total: 29.5ms	remaining: 24.9s
150:	learn: 1.4399277	total: 4.32s	remaining: 19.8s
300:

,Id,team_goals,opp_goals
0,M034984_Seychelles,2,1
1,M034984_Mauritius,1,2
2,M034985_Comoros,1,2
3,M034985_Maldives,2,1
4,M034986_Réunion,2,1


,team_goals,opp_goals
count,42422.000000,42422.000000
mean,1.513672,1.508439
std,0.998308,0.995718
min,0.000000,0.000000
25%,1.000000,1.000000
50%,1.000000,1.000000
75%,2.000000,2.000000
max,4.000000,4.000000



distribusi skor

team_goals
team_goals
1    19189
2    12473
0     4914
3     3306
4     2540
Name: count, dtype: int64

opp_goals
opp_goals
1    19181
2    12499
0     4969
3     3280
4     2493
Name: count, dtype: int64

distribusi pasangan skor


,,count
team_goals,opp_goals,
1,2,10920
2,1,10874
1,1,6020
4,0,1904
0,4,1875
3,1,1580
0,3,1562
3,0,1554
1,3,1548


## 13. Notes

Kalau hasil Kaggle naik: classifier membantu memilih outcome yang lebih konsisten.

Kalau Kaggle turun:
1. coba file backup roundclip,
2. bandingkan distribusi skor dengan submission 2.90-an,
3. next pure-ML upgrade adalah pseudo-sequential update test memakai prediksi tahap pertama.